In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
print ("librares imported")


librares imported


In [3]:
import os

os.getcwd()

'c:\\Users\\valer\\OneDrive\\Desktop\\Dissertation Project\\DissertationApp'

In [6]:
os.listdir()

['.$Backend layer Architecture.drawio.bkp',
 '.$Development Environment Diagram.drawio.bkp',
 '.$Frontend layer Architecture Diagram.drawio.bkp',
 '.$Frontend-Backend Communication Architecture.drawio.bkp',
 '.$ML Pipeline Diagram.drawio.bkp',
 '.$Security Architecture Diagram.drawio.bkp',
 '.$Sequence diagram reservation forecast.drawio.bkp',
 '.$Syetem Architecture Diagram.drawio.bkp',
 '.$USE CASE DIAGRAM PLUEPRINT.drawio.bkp',
 '.$USE CASE DIAGRAM.png.bkp',
 '.$User Login Sequence.drawio.bkp',
 '.git',
 '.gitignore',
 'Architecture Diagrams',
 'backend',
 'dbpass.txt',
 'frontend',
 'model_comparison.ipynb',
 'node_modules',
 'package-lock.json',
 'package.json',
 'README.md',
 'restaurant_demand_features.csv',
 'Sequence Diagrams']

In [7]:
import pandas as pd

df = pd.read_csv("restaurant_demand_features.csv")

df.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary,day_of_week,month,week_of_year,day_of_month,is_weekend
0,2024-01-02,0,5,7,12,118,2,0,3,5,114,Tuesday,1,1,2,0
1,2024-01-03,0,16,15,31,116,6,0,5,11,115,Wednesday,1,1,3,0
2,2024-01-04,0,14,20,34,113,6,0,5,11,112,Thursday,1,1,4,0
3,2024-01-05,0,19,60,79,112,21,0,7,28,108,Friday,1,1,5,0
4,2024-01-06,0,43,38,81,108,14,0,13,27,106,Saturday,1,1,6,1


In [8]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.columns

Rows: 747
Columns: 16


Index(['date', 'same_day_covers', 'walk_in_covers', 'advance_covers',
       'total_covers', 'avg_duration_covers_summary', 'advance_bookings',
       'same_day_bookings', 'walk_in_bookings', 'total_bookings',
       'avg_duration_bookings_summary', 'day_of_week', 'month', 'week_of_year',
       'day_of_month', 'is_weekend'],
      dtype='object')

In [9]:
df = pd.read_csv("restaurant_demand_features.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Convert date and sort by date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

# Create numeric day of week from date
df["day_of_week_num"] = df["date"].dt.dayofweek

# Create historical lag and rolling average features
history_columns = [
    "total_covers",
    "same_day_covers",
    "walk_in_covers",
    "advance_covers",
    "total_bookings",
    "avg_duration_covers_summary"
]

for col in history_columns:
    df[f"{col}_lag_1"] = df[col].shift(1)
    df[f"{col}_lag_7"] = df[col].shift(7)
    df[f"{col}_lag_14"] = df[col].shift(14)
    df[f"{col}_avg_7"] = df[col].shift(1).rolling(7).mean()
    df[f"{col}_avg_30"] = df[col].shift(1).rolling(30).mean()

df.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,...,total_bookings_lag_1,total_bookings_lag_7,total_bookings_lag_14,total_bookings_avg_7,total_bookings_avg_30,avg_duration_covers_summary_lag_1,avg_duration_covers_summary_lag_7,avg_duration_covers_summary_lag_14,avg_duration_covers_summary_avg_7,avg_duration_covers_summary_avg_30
0,2024-01-02,0,5,7,12,118,2,0,3,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-03,0,16,15,31,116,6,0,5,11,...,5.0,NaN,NaN,NaN,NaN,118.0,NaN,NaN,NaN,NaN
2,2024-01-04,0,14,20,34,113,6,0,5,11,...,11.0,NaN,NaN,NaN,NaN,116.0,NaN,NaN,NaN,NaN
3,2024-01-05,0,19,60,79,112,21,0,7,28,...,11.0,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN
4,2024-01-06,0,43,38,81,108,14,0,13,27,...,28.0,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN


In [10]:
target_col = "total_covers"

feature_cols = [
    "day_of_week_num",
    "month",
    "week_of_year",
    "day_of_month",
    "is_weekend",

    "total_covers_lag_1",
    "total_covers_lag_7",
    "total_covers_lag_14",
    "total_covers_avg_7",
    "total_covers_avg_30",

    "same_day_covers_avg_7",
    "same_day_covers_avg_30",
    "walk_in_covers_avg_7",
    "walk_in_covers_avg_30",
    "advance_covers_avg_7",
    "advance_covers_avg_30",

    "total_bookings_avg_7",
    "total_bookings_avg_30",
    "avg_duration_covers_summary_avg_7",
    "avg_duration_covers_summary_avg_30"
]

model_data = df.dropna(subset=feature_cols + [target_col]).copy()

print("Rows available for comparison:", len(model_data))
print("Features used:", len(feature_cols))

Rows available for comparison: 717
Features used: 20


In [11]:
split_index = int(len(model_data) * 0.8)

train = model_data.iloc[:split_index]
test = model_data.iloc[split_index:]

X_train = train[feature_cols]
y_train = train[target_col]

X_test = test[feature_cols]
y_test = test[target_col]

print("Training rows:", len(train))
print("Testing rows:", len(test))

Training rows: 573
Testing rows: 144


In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

models = {
    "Baseline Model": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=42,
        max_depth=6
    ),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2
    )
}

results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = root_mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model tested": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 3)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="RMSE").reset_index(drop=True)

results_df

,Model tested,MAE,RMSE,R2 Score
0,Random Forest Regressor,13.49,18.16,0.304
1,Decision Tree Regressor,14.30,20.87,0.081
2,Baseline Model,16.50,21.78,-0.001
3,Linear Regression,18.78,25.56,-0.378
